In [1]:
# 0. Preparation: Add the 'code/' directory to the Python import path.
import sys
from pathlib import Path

# Locate the 'code' directory in the parent folder of the current notebook.
EVAL_CODE = (Path.cwd().parent / "code").resolve()
# If the path is not found, force the absolute path to the Evaluation/code directory.
if not (EVAL_CODE / "evaluation_utils.py").exists():
    EVAL_CODE = Path(r"E:/IT_SPACES/AI/ZoomCamp/LLM/04/2026/Evaluation/code")
# Insert the path at the start of sys.path so Python can find the custom modules.
sys.path.insert(0, str(EVAL_CODE))

# Import required tools, instructions, and classes from the evaluation_utils module.
from gitsource import GithubRepositoryDataReader
from evaluation_utils import (
    GEMMA_DATA_GEN_INSTRUCTIONS,
    Questions,
    get_default_model,
    get_openai_client,
    get_prompt_tokens,
    llm_structured,
)

# 1. Load lesson documents: Fetch markdown files from GitHub based on a specific commit.
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path, # Filter to include only files in the 'lessons' folder.
)
# Parse the read files and store them as a list of document objects.
documents = [file.parse() for file in reader.read()]
# Store the instructions used for generating questions.
data_gen_instructions = GEMMA_DATA_GEN_INSTRUCTIONS

# 2. Connect to Cerebras client: Prepare for communication with the API server.
client = get_openai_client(local=True)
# Retrieve the model name (e.g., gemma-4-31b) specified in the configuration.
model = get_default_model(client, local=True)
# Print status to verify the backend, model, and the total number of loaded documents.
print(f"backend={client.base_url} model={model} docs={len(documents)}")

backend=https://api.cerebras.ai/v1/ model=gemma-4-31b docs=72


In [2]:
# Q1: Calculate the average input token count for the 3 target lesson files.

# 1. Define the list of target file paths for question generation.
target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

# 2. Prepare an empty bucket (list) to store the measured input token count for each file.
input_tokens_list = []

# 3. Iterate through all documents and process only the files specified in target_files.
for doc in documents:
    # Skip documents that are not part of the target file list.
    if doc["filename"] not in target_files:
        continue

    # Create a prompt for the LLM by combining the filename, content, and generation instructions.
    user_prompt = (
        f"Filename: {doc['filename']}\n"
        f"Content: {doc['content']}\n\n"
        f"{data_gen_instructions}"
    )

    # 4. Call the LLM. 
    # 'llm_structured' sends the prompt to the server and receives a 'usage' package containing consumption details.
    _, usage = llm_structured(
        client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=model,
    )
    
    # 5. 'usage' is the detailed receipt from the server; 'get_prompt_tokens' is a tool to extract only the input tokens.
    # We create a variable named 'tokens' to hold this count and add it to our tracking bucket (input_tokens_list).
    input_tokens_list.append(get_prompt_tokens(usage))

# 6. Sum all token counts in the bucket and divide by 3 to calculate the average.
average_tokens = sum(input_tokens_list) / len(input_tokens_list)

# 7. Print the results to verify the final average input token count.
print(f"input tokens per file: {input_tokens_list}")
print(f"Q1 average input tokens: {average_tokens}")

input tokens per file: [1110, 1352, 1812]
Q1 average input tokens: 1424.6666666666667


In [3]:
# Q2 setup: ground truth CSV + text search index (same as HW2)
import pandas as pd
from gitsource import chunk_documents
from minsearch import Index

GROUND_TRUTH_CSV = (Path.cwd() / "data" / "ground-truth.csv").resolve()
if not GROUND_TRUTH_CSV.exists():
    GROUND_TRUTH_CSV = Path(r"E:/IT_SPACES/AI/ZoomCamp/LLM/04/2026/Evaluation/LLM_04_HW/data/ground-truth.csv")

df_ground_truth = pd.read_csv(GROUND_TRUTH_CSV)
ground_truth = df_ground_truth.to_dict(orient="records")

chunks = chunk_documents(documents, size=2000, step=1000)
ms_text = Index(text_fields=["content"], keyword_fields=["filename"])
ms_text.fit(chunks)

def text_search(query, num_results=5):
    return ms_text.search(query=query, num_results=num_results)

print(f"ground_truth={len(ground_truth)} chunks={len(chunks)}")

ground_truth=360 chunks=295


In [4]:
# Retrieve the 'question' value from the first entry of the 
# ground_truth dataset and store it in the 'query' variable.
query = ground_truth[0]["question"]

# Pass the query to the 'text_search' function to perform the search 
# and store the resulting list in 'search_results'.
search_results = text_search(query)

# Extract the 'filename' from the first (index 0) item in the 
# 'search_results' list and store it in 'first_result_filename'.
first_result_filename = search_results[0]["filename"]

# Print the identified filename to the console.
print(f"First result filename: {first_result_filename}")

First result filename: 01-agentic-rag/lessons/03-rag.md


In [5]:
# Q3 setup: Create a vector search index (using the same embedder/model as HW2).
import numpy as np
from minsearch import VectorSearch

# Set the directory path where the embedder files are located.
EMBED_DIR = (Path.cwd().parents[3] / "02" / "2026" / "Vector_Search" / "embed").resolve()

# If the file does not exist in the default path, force the path to the
# absolute location on the E drive.
if not (EMBED_DIR / "embedder.py").exists():
    EMBED_DIR = Path(r"E:/IT_SPACES/AI/ZoomCamp/LLM/02/2026/Vector_Search/embed")

# Define the specific directory path where the model is stored.
MODEL_DIR = EMBED_DIR / "models" / "Xenova" / "all-MiniLM-L6-v2"
# Insert the embedder directory into the system path so it can be imported.
sys.path.insert(0, str(EMBED_DIR))

# Import the 'Embedder' class to handle vectorization.
from embedder import Embedder

# Initialize the embedder object using the specified model.
embed_model = Embedder(path=MODEL_DIR)
# Extract only the content field from each chunk to prepare for encoding.
chunk_contents = [c["content"] for c in chunks]

# Encode the text into vectors in batches of 50 to optimize memory usage.
embed_parts = [
    embed_model.encode_batch(chunk_contents[i : i + 50])
    for i in range(0, len(chunk_contents), 50)
]
# Vertically stack the batched vectors to create a single matrix (X).
X = np.vstack(embed_parts)

# Create a VectorSearch index object, setting 'filename' as a keyword field.
ms_vector = VectorSearch(keyword_fields=["filename"])
# Fit the index using the vector matrix (X) and the original chunk data.
ms_vector.fit(X, chunks)

# Define a function to perform search by converting the query into a vector.
def vector_search(query, num_results=5):
    return ms_vector.search(embed_model.encode(query), num_results=num_results)

# Print a status message to verify the index is ready and show the matrix shape.
print(f"vector index ready: {X.shape}")

vector index ready: (295, 384)


In [8]:
# Use the first question from the ground_truth dataset to perform a vector search.
# Store the search results in the 'vector_results' list using the 'vector_search' function.
vector_results = vector_search(query)

# Extract the 'filename' from the first (index 0) item in the 
# 'vector_results' list and store it in 'vector_result_filename'.
vector_result_filename = vector_results[0]["filename"]

# Print the final identified filename to the console.
print(f"Vector search result filename: {vector_result_filename}")

Vector search result filename: 01-agentic-rag/lessons/01-intro.md


In [9]:
# Q4
# Initialize the variable to store the count of successful matches (hits).
hit_count = 0

# Iterate through each entry in the ground_truth dataset to evaluate performance.
for entry in ground_truth:
    # 1. Retrieve the question and perform a text search.
    question = entry["question"]
    results = text_search(question, num_results=5)
    
    # 2. Extract only the filenames from the search results.
    result_filenames = [r["filename"] for r in results]
    
    # 3. Check if the correct filename (ground truth) is in the search results.
    if entry["filename"] in result_filenames:
        hit_count += 1

# 4. Calculate the Hit Rate by dividing the total hits by the number of questions.
hit_rate = hit_count / len(ground_truth)

# Print the final Hit Rate result.
print(f"Hit Rate: {hit_rate:.2f}")

Hit Rate: 0.76


In [10]:
# Q5
# Initialize the total reciprocal rank.
total_rr = 0.0

# Iterate through each entry in the ground_truth dataset.
for entry in ground_truth:
    # 1. Perform vector search.
    results = vector_search(entry["question"], num_results=5)
    
    # 2. Find the rank (1-based index) of the correct filename.
    rank = 0
    for i, r in enumerate(results):
        if r["filename"] == entry["filename"]:
            rank = i + 1
            break
            
    # 3. If found, add the reciprocal rank (1/rank) to the total.
    if rank > 0:
        total_rr += 1 / rank

# 4. Calculate the Mean Reciprocal Rank (MRR).
mrr = total_rr / len(ground_truth)

# Print the final MRR result.
print(f"MRR: {mrr:.2f}")

MRR: 0.55


In [13]:
# 1. This function performs Hybrid Search using RRF (Reciprocal Rank Fusion).
# It merges results from both vector and keyword searches to improve ranking.
def hybrid_search(query, k=60):
    # Retrieve top 5 results from both vector and keyword search methods.
    v_results = vector_search(query, num_results=5)
    t_results = text_search(query, num_results=5)
    
    # Create a 'bucket' to store cumulative scores for each document.
    fused_scores = {}
    
    # Calculate scores based on the rank in vector search and add to the bucket.
    for rank, hit in enumerate(v_results):
        doc_id = hit["filename"]
        # Score formula: 1 / (k + rank + 1). Higher k makes the gap between ranks smaller.
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + 1 / (k + rank + 1)
        
    # Do the same for keyword search results.
    for rank, hit in enumerate(t_results):
        doc_id = hit["filename"]
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + 1 / (k + rank + 1)
        
    # Sort the files by their total fused score in descending order and return them.
    reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return reranked

# 2. This function evaluates the performance (MRR) of the hybrid search.
def evaluate_hybrid(k_value):
    total_rr = 0.0 # Variable to accumulate the reciprocal ranks.
    
    # Iterate through every question in your ground_truth dataset.
    for entry in ground_truth:
        # Perform hybrid search using the specific k value provided.
        results = hybrid_search(entry["question"], k=k_value)
        
        rank = 0
        # Check the rank (position) of the correct filename in the search results.
        for i, r in enumerate(results):
            if r[0] == entry["filename"]:
                rank = i + 1 # Add 1 because index starts at 0.
                break
        
        # If the correct answer is found, add its reciprocal (1/rank) to the total.
        if rank > 0:
            total_rr += 1 / rank
            
    # Return the average MRR score by dividing by the total number of questions.
    return total_rr / len(ground_truth)

# 3. Iterate through the requested k values to compare performance.
for k in [1, 50, 100, 200]:
    # Calculate the MRR for the current k value.
    mrr = evaluate_hybrid(k_value=k)
    # Print the results so you can easily identify which k yields the best MRR.
    print(f"k={k}: MRR={mrr:.2f}")

k=1: MRR=0.65
k=50: MRR=0.64
k=100: MRR=0.64
k=200: MRR=0.64
